# 0. Imports

In [18]:
from pathlib import Path

import numpy as np
import pandas as pd
import polars as pl
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset

# 1. Download and Store Dataset

In [19]:
def ingest_reddit_data(subreddit_key: str, n_rows: int = 1_000_000, force_rerun: bool = False) -> Path:
    """
    Orchestrates the ETL process for a specific subreddit's comment data.
    
    Args:
        subreddit_key: Dictionary key from 'splits' (e.g., 'changemyview').
        n_rows: Maximum records to process for the local sample.
        force_rerun: If True, bypasses existence check and overwrites existing parquet file.
        
    Returns:
        Path to the processed Parquet file.
    """
    out_path = Path(f"data/processed/{subreddit_key}_sample.parquet")
    
    # Idempotency check: Skip heavy network I/O if the target file is already present
    if out_path.exists() and not force_rerun:
        print(f"Skipping ingestion: Local cache found at {out_path}")
        return out_path

    out_path.parent.mkdir(parents=True, exist_ok=True)

    # Define schema subset based on downstream analytical requirements
    feature_cols = [
        "author", "body", "created_utc", "id", "link_id", "name",
        "parent_id", "score", "controversiality", "total_awards_received"
    ]
    splits = {
    'changemyview': 'data/changemyview-*-of-*.parquet',
    }

    print(f"Streaming data from HuggingFace for: r/{subreddit_key}...")
    
    # Execute lazy-evaluated ETL pipeline
    try:
        (
            pl.scan_parquet(f"hf://datasets/HuggingFaceGECLM/REDDIT_comments/{splits[subreddit_key]}")
            .select(feature_cols)
            # Filter out deleted/removed content to maintain high data quality for NLP tasks
            .filter(~pl.col("body").is_in(["[deleted]", "[removed]"]))
            .limit(n_rows)
            # Stream directly to disk using ZSTD to balance compression ratio and write speed
            .sink_parquet(out_path, compression="zstd")
        )
        print(f"Successfully wrote {n_rows} rows to {out_path}")
    except KeyError:
        raise ValueError(f"Subreddit '{subreddit_key}' not found in defined splits.")
    except Exception as e:
        print(f"Pipeline failed: {e}")
        raise

    return out_path

# --- Execution Control ---
# Toggle 'force_rerun' if the upstream data schema changes or a larger sample is needed
OUT = ingest_reddit_data("changemyview", n_rows=1_000_000, force_rerun=False)

Skipping ingestion: Local cache found at data/processed/changemyview_sample.parquet


# 2. Load Data from Parquet File

In [20]:
df = (
    # Scan the metadata and define the lazy query plan
    pl.scan_parquet("data/processed/changemyview_clean_head.parquet")
      # Constrain sample size for rapid local prototyping
      .head(10_000)
      # Trigger execution and load into memory
      .collect()
      # Bridge to Pandas for ecosystem compatibility
      .to_pandas()
)

# 3. Create Train and Test Dataset

### 3.1 Data Preprocessing, Temporal Splitting & Metadata Mapping

In [21]:
# --- 1. Data Cleaning & Type Casting ---

# Ensure text integrity by removing null observations in the primary feature
df = df.dropna(subset=["body"])

# Filter out anonymous/deleted accounts to maintain attribution quality
df = df[df["author"] != "[deleted]"]

# Normalize timestamps: Convert raw strings to numeric Unix seconds, then to datetime objects
# 'coerce' handles malformed strings by returning NaT, preventing pipeline crashes
df["created_utc"] = pd.to_numeric(df["created_utc"], errors="coerce")
df["date"] = pd.to_datetime(df["created_utc"], unit="s", errors="coerce")


# --- 2. Temporal Train/Test Split ---

# Use a temporal 80/20 split rather than a random shuffle to prevent 'look-ahead' bias.
# This simulates a real-world scenario where we predict future comments based on past data.
cutoff = df["created_utc"].quantile(0.8)

df_train = df[df["created_utc"] <= cutoff].copy()
df_test  = df[df["created_utc"] > cutoff].copy()


# --- 3. Metadata Mapping (Lookup Tables) ---

# Create lightweight author lookups for efficient O(1) retrieval.
# Mappings are scoped strictly within splits to enforce isolation and prevent leakage.
id2author_train = df_train.set_index("id")["author"].to_dict()
id2author_test  = df_test.set_index("id")["author"].to_dict()

### 3.2 Interaction Network Construction

In [22]:
def build_reply_pairs(df_split, id2author):
    """
    Constructs a positive interaction dataset by mapping comments to their parent authors.
    Filters for comment-to-comment replies and removes self-interactions.
    """
    # Reddit 'parent_id' prefixes: t1 = Comment, t3 = Link/Post.
    # We restrict analysis to comment-to-comment interactions to capture conversational dynamics.
    parent_comment_ids = df_split["parent_id"].astype(str)
    is_comment_reply = parent_comment_ids.str.startswith("t1_")
    df_r = df_split[is_comment_reply].copy()

    # Extract the raw 36-base ID by stripping the 't1_' type prefix for join compatibility
    df_r["parent_key"] = df_r["parent_id"].str.replace("^t1_", "", regex=True)

    # Resolve parent author identities via the provided lookup table (O(1) mapping)
    df_r["parent_author"] = df_r["parent_key"].map(id2author)

    # --- Data Integrity & Quality Filtering ---
    # 1. Drop replies where the parent comment falls outside the current split (boundary integrity)
    df_r = df_r.dropna(subset=["parent_author"])
    # 2. Exclude self-replies to ensure we only model interpersonal interactions
    df_r = df_r[df_r["author"] != df_r["parent_author"]]

    # Feature selection and renaming to standard (u, v) graph notation
    pairs_pos = df_r[["author", "parent_author", "created_utc", "link_id", "id", "parent_key"]].copy()
    pairs_pos = pairs_pos.rename(columns={
        "author": "u", 
        "parent_author": "v", 
        "id": "u_comment_id", 
        "parent_key": "v_comment_id"
    })
    
    # Label as positive instances for downstream binary classification
    pairs_pos["y"] = 1
    return pairs_pos

# Generate interaction sets; scoped within splits to prevent data leakage
pos_train = build_reply_pairs(df_train, id2author_train)
pos_test  = build_reply_pairs(df_test, id2author_test)

print(f"Positive samples - Train: {len(pos_train):,} | Test: {len(pos_test):,}")

Positive samples - Train: 4,356 | Test: 928


In [23]:
# --- Graph Diagnostics: Sparsity & Degree Distribution ---

# Calculate the ratio of users who engaged in at least one reply
pos_users = set(pos_train["u"]) | set(pos_train["v"])
all_users = set(df_train["author"].dropna().unique())
print(f"Engagement Coverage: {len(pos_users)} / {len(all_users)} users with interactions")

# Analyze the 'Out-Degree' (number of replies sent per user)
print("\nReplies per user statistics:")
print(pos_train.groupby("u").size().describe())

Engagement Coverage: 999 / 1321 users with interactions

Replies per user statistics:
count    884.000000
mean       4.927602
std        9.560404
min        1.000000
25%        1.000000
50%        2.000000
75%        5.000000
max      123.000000
dtype: float64


### 3.3 Negative Sampling Strategy

In [24]:
def build_hard_negatives(df_split, pos_pairs, k_per_pos=2, seed=42):
    """
    Generates 'hard' negative samples for link prediction by identifying potential 
    interactions that did NOT occur within the same discussion thread context.
    """
    # Initialize a BitGenerator for reproducible stochastic sampling
    rng = np.random.default_rng(seed)

    # 1) Contextual Mapping: Identify all active participants per discussion thread (link_id).
    # This defines our 'closed-world' candidate pool for each observation.
    thread_users = (
        df_split.groupby("link_id")["author"]
        .apply(lambda s: set(s.dropna()))
        .to_dict()
    )

    # 2) Network Topology: Extract existing interaction edges in (u, v) space.
    # We treat edges as symmetric to prevent sampling reciprocal replies as negatives,
    # which would introduce label noise.
    reply_edges = set(zip(pos_pairs["u"], pos_pairs["v"]))
    reply_edges_sym = reply_edges | {(v, u) for (u, v) in reply_edges}

    neg_rows = []
    # Project to minimal feature set to reduce overhead during iteration
    pos_pairs_small = pos_pairs[["u", "v", "link_id"]].copy()

    for u, v, link_id in pos_pairs_small.itertuples(index=False):
        users = list(thread_users.get(link_id, []))
        if len(users) <= 1:
            continue

        # Candidate Filtering: 
        # Target users in the same thread (high-signal 'hard' negatives) excluding the source 'u'
        cand = [x for x in users if x != u]
        if not cand:
            continue

        # Collision Avoidance: Remove candidates where a ground-truth interaction (u, x) exists
        cand = [x for x in cand if (u, x) not in reply_edges_sym]
        if not cand:
            continue

        # Stochastic Sampling: Select 'k' negatives per positive to maintain class ratio
        take = min(k_per_pos, len(cand))
        sampled = rng.choice(cand, size=take, replace=False)

        for x in sampled:
            neg_rows.append((u, x, link_id, 0))

    return pd.DataFrame(neg_rows, columns=["u", "v", "link_id", "y"])

# --- Dataset Assembly ---

# Generate split-specific negatives to ensure no data leakage across the temporal boundary
neg_train = build_hard_negatives(df_train, pos_train, k_per_pos=2)
neg_test  = build_hard_negatives(df_test,  pos_test,  k_per_pos=2)

# Final Concatenation & Shuffling:
# We combine positive (y=1) and negative (y=0) instances, then shuffle to ensure 
# gradient descent isn't biased by class-ordered mini-batches.
train_pairs = pd.concat([pos_train[["u","v","link_id","y"]], neg_train], ignore_index=True).sample(frac=1, random_state=42)
test_pairs  = pd.concat([pos_test[["u","v","link_id","y"]],   neg_test],  ignore_index=True).sample(frac=1, random_state=42)

# Diagnostic: Verify class balance (standard ratio is 1:k)
print("Training Class Distribution:\n", train_pairs["y"].value_counts())
print("Testing Class Distribution:\n", test_pairs["y"].value_counts())

Training Class Distribution:
 y
0    8557
1    4356
Name: count, dtype: int64
Testing Class Distribution:
 y
0    1795
1     928
Name: count, dtype: int64


In [25]:
# --- Unit Tests: Sampling Integrity ---

# Ensure no user is paired with themselves (self-loops)
assert (train_pairs["u"] != train_pairs["v"]).all(), "Found self-interactions in training set"

# Verify that negative samples do not overlap with ground-truth positive replies
real_edges = set(zip(pos_train["u"], pos_train["v"]))
assert not any(
    (u, v) in real_edges or (v, u) in real_edges
    for u, v in zip(neg_train["u"], neg_train["v"])
), "Negative samples contain real interaction edges"

### 3.4 User Textual Profile Construction

In [26]:
# --- 1. Corpus Preparation & Leakage Prevention ---

# Isolate training and testing text to ensure that future comments do not 
# influence the historical representations of users in the training set.
df_train_text = df_train.dropna(subset=["body", "id"]).copy()
df_test_text = df_test.dropna(subset=["body", "id"]).copy()

# (Optional) Heuristic: Filter for active users to ensure embeddings have 
# sufficient signal (min 5 observations). 
# df_train_text = df_train_text.groupby("id").filter(lambda g: len(g) >= 5)

# --- 2. Temporal Aggregation (Feature Engineering) ---

# Construct a profile for each author.
# We join the most recent comments to capture the user's current interests/voice.
user_text_train = (
    df_train_text
    .sort_values("created_utc")             # Enforce chronology to correctly identify the 'tail'
    .groupby("author")["body"]
    # Hyperparameter: Concatenating the last 10 comments balances context vs. sequence length
    .apply(lambda s: " ".join(s.tail(10)))  
)

user_text_test = (
    df_test_text
    .sort_values("created_utc")
    .groupby("author")["body"]
    .apply(lambda s: " ".join(s.tail(10)))
)

# Convert to hash maps (dict) for O(1) lookup performance during the mapping phase
user_text_dict_train = user_text_train.to_dict()
user_text_dict_test = user_text_test.to_dict()

# --- 3. Coverage Analysis (Data Integrity Check) ---

# Quantify the 'Cold-Start' issue: users in the interaction pairs who lack 
# textual history. Significant missingness here indicates a sampling mismatch.
missing_train = train_pairs["u"].map(user_text_dict_train).isna().mean()
print(f"Missing text profile ratio (Train - Source User): {missing_train:.2%}")

missing_test = test_pairs["u"].map(user_text_dict_test).isna().mean()
print(f"Missing text profile ratio (Test - Source User): {missing_test:.2%}")

Missing text profile ratio (Train - Source User): 0.00%
Missing text profile ratio (Test - Source User): 0.00%


In [27]:
# --- 1. Prepare Data Containers ---

# Create copies to prevent SettingWithCopy warnings and isolate split changes
train_pairs = train_pairs.copy()
test_pairs  = test_pairs.copy()

def attach_text(pairs, user_text_dict):
    """Adds historical text for source (u) and target (v) users."""
    
    # Map text profiles to user IDs
    pairs["text_u"] = pairs["u"].map(user_text_dict)
    pairs["text_v"] = pairs["v"].map(user_text_dict)
    
    # Remove observations missing text for either user to ensure a complete feature set
    return pairs.dropna(subset=["text_u", "text_v"])

# --- 2. Execute Merge & Cleanup ---

train_pairs_txt = attach_text(train_pairs, user_text_dict_train)
test_pairs_txt  = attach_text(test_pairs,  user_text_dict_test)

# --- 3. Progress Check ---

# Log row counts to monitor data loss during the mapping/dropping process
print(f"Train Retention: {len(train_pairs):,} -> {len(train_pairs_txt):,}")
print(f"Test Retention:  {len(test_pairs):,} -> {len(test_pairs_txt):,}")

Train Retention: 12,913 -> 12,913
Test Retention:  2,723 -> 2,723


### 3.5 Save Train and Test Datasets

In [28]:
train_pairs_txt.to_parquet("data/processed/train_pairs_txt.parquet", index=False)
test_pairs_txt.to_parquet("data/processed/test_pairs_txt.parquet", index=False)

# 5. Train CNN

### 5.1 Tokenize data

In [31]:
import re
from collections import Counter

TOKEN_RE = re.compile(r"[A-Za-z']+")

def tokenize(text: str):
    return TOKEN_RE.findall(text.lower())

MAX_VOCAB = 50_000
MIN_FREQ = 2

# Build vocab from TRAIN texts only (no leakage)
counter = Counter()
for t in train_pairs_txt["text_u"].tolist():
    counter.update(tokenize(t))
for t in train_pairs_txt["text_v"].tolist():
    counter.update(tokenize(t))

# Special tokens
PAD = "<pad>"
UNK = "<unk>"

vocab = {PAD: 0, UNK: 1}
for w, c in counter.most_common(MAX_VOCAB):
    if c < MIN_FREQ:
        break
    vocab[w] = len(vocab)

pad_id = vocab[PAD]
unk_id = vocab[UNK]

print("Vocab size:", len(vocab))


Vocab size: 19854


### 5.2 Create Dataset

In [32]:
MAX_LEN = 256  # truncate long user docs

def encode(text: str):
    ids = [vocab.get(w, unk_id) for w in tokenize(text)]
    return ids[:MAX_LEN]

class PairDataset(Dataset):
    def __init__(self, df_pairs):
        self.u_ids = df_pairs["u_id"].tolist()
        self.v_ids = df_pairs["v_id"].tolist()
        self.u_texts = df_pairs["text_u"].tolist()
        self.v_texts = df_pairs["text_v"].tolist()
        self.y = df_pairs["y"].astype(float).tolist()

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        return (
            self.u_ids[idx],
            self.v_ids[idx],
            encode(self.u_texts[idx]),
            encode(self.v_texts[idx]),
            self.y[idx],
        )
    
def collate_fn(batch):
    u_ids, v_ids, u_seqs, v_seqs, ys = zip(*batch)

    u_lens = torch.tensor([len(s) for s in u_seqs], dtype=torch.long)
    v_lens = torch.tensor([len(s) for s in v_seqs], dtype=torch.long)

    max_u = max(u_lens).item()
    max_v = max(v_lens).item()

    u = torch.full((len(batch), max_u), pad_id, dtype=torch.long)
    v = torch.full((len(batch), max_v), pad_id, dtype=torch.long)

    for i, s in enumerate(u_seqs):
        u[i, :len(s)] = torch.tensor(s, dtype=torch.long)
    for i, s in enumerate(v_seqs):
        v[i, :len(s)] = torch.tensor(s, dtype=torch.long)

    y = torch.tensor(ys, dtype=torch.float32)
    # u_ids = torch.tensor(u_ids, dtype=torch.long)
    # v_ids = torch.tensor(v_ids, dtype=torch.long)

    return list(u_ids), list(v_ids), u, v, y


In [33]:
BATCH_SIZE = 128

train_ds = PairDataset(train_pairs_txt)
test_ds  = PairDataset(test_pairs_txt)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_fn, num_workers=0)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn, num_workers=0)


KeyError: 'u_id'

### 5.3 Create Siamese CNN

In [ ]:
class TextCNNEncoder(nn.Module):
    def __init__(self, vocab_size, emb_dim=128, num_filters=128, kernel_sizes=(3,4,5), out_dim=128, pad_idx=0, dropout=0.2):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, emb_dim, padding_idx=pad_idx)
        self.convs = nn.ModuleList([
            nn.Conv1d(in_channels=emb_dim, out_channels=num_filters, kernel_size=k)
            for k in kernel_sizes
        ])
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(num_filters * len(kernel_sizes), out_dim)

    def forward(self, x):
        # x: [B, T]
        emb = self.embedding(x)            # [B, T, E]
        emb = emb.transpose(1, 2)          # [B, E, T] for Conv1d

        conv_outs = []
        for conv in self.convs:
            h = F.relu(conv(emb))          # [B, F, T-k+1]
            h = F.max_pool1d(h, kernel_size=h.size(2)).squeeze(2)  # [B, F]
            conv_outs.append(h)

        h = torch.cat(conv_outs, dim=1)    # [B, F*len(K)]
        h = self.dropout(h)
        h = self.fc(h)                     # [B, out_dim]
        h = F.normalize(h, p=2, dim=1)     # unit vectors
        return h

class SiameseCNN(nn.Module):
    def __init__(self, encoder: nn.Module, scale=10.0):
        super().__init__()
        self.encoder = encoder
        self.scale = scale  # scales cosine to logits

    def forward(self, u, v):
        eu = self.encoder(u)
        ev = self.encoder(v)
        cos = (eu * ev).sum(dim=1)         # cosine, since normalized
        logits = self.scale * cos          # logit
        return logits

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

encoder = TextCNNEncoder(
    vocab_size=len(vocab),
    emb_dim=128,
    num_filters=128,
    kernel_sizes=(3,4,5),
    out_dim=128,
    pad_idx=pad_id,
    dropout=0.2,
)

model = SiameseCNN(encoder, scale=10.0).to(device)

### 5.4 Train the CNN and Evaluate

In [ ]:
criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=2e-3, weight_decay=1e-2)

@torch.no_grad()
def eval_auc(model, loader):
    model.eval()
    ys, ps = [], []
    for u_ids, v_ids, u, v, y in loader:   # <- changed
        u, v = u.to(device), v.to(device)
        logits = model(u, v)
        prob = torch.sigmoid(logits).cpu().numpy()
        ys.extend(y.numpy())
        ps.extend(prob)

    ys = np.array(ys)
    ps = np.array(ps)

    order = np.argsort(ps)
    ys_sorted = ys[order]
    n_pos = ys_sorted.sum()
    n_neg = len(ys_sorted) - n_pos
    if n_pos == 0 or n_neg == 0:
        return float("nan")
    ranks = np.arange(1, len(ys_sorted) + 1)
    rank_sum_pos = ranks[ys_sorted == 1].sum()
    auc = (rank_sum_pos - n_pos*(n_pos+1)/2) / (n_pos*n_neg)
    return float(auc)

def train_one_epoch(model, loader):
    model.train()
    total_loss = 0.0
    for u_ids, v_ids, u, v, y in loader:   # <- changed
        u, v, y = u.to(device), v.to(device), y.to(device)

        optimizer.zero_grad(set_to_none=True)
        logits = model(u, v)
        loss = criterion(logits, y)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()

        total_loss += loss.item() * len(y)
    return total_loss / len(loader.dataset)

EPOCHS = 5
for epoch in range(1, EPOCHS + 1):
    loss = train_one_epoch(model, train_loader)
    auc = eval_auc(model, test_loader)
    print(f"epoch={epoch}  loss={loss:.4f}  test_auc={auc:.4f}")


epoch=1  loss=1.1136  test_auc=0.6024
epoch=2  loss=0.8880  test_auc=0.6039
epoch=3  loss=0.8039  test_auc=0.6012
epoch=4  loss=0.7397  test_auc=0.6089
epoch=5  loss=0.6749  test_auc=0.6108


In [ ]:
@torch.no_grad()
def save_predictions_parquet(model, loader, out_path: str):
    model.eval()
    rows = []

    for u_ids, v_ids, u, v, y in loader:
        u, v = u.to(device), v.to(device)
        logits = model(u, v)
        probs = torch.sigmoid(logits).cpu().numpy()

        # u_ids_np = u_ids.numpy()
        # v_ids_np = v_ids.numpy()
        y_np = y.numpy()

        for ui, vi, yt, sc in zip(u_ids, v_ids, y_np, probs):
            rows.append({
                "u_id": ui,
                "v_id": vi,
                "y": int(yt),
                "score": float(sc),
            })

    df_out = pd.DataFrame(rows)
    df_out.to_parquet(out_path, index=False)
    return df_out

pred_df = save_predictions_parquet(model, test_loader, "predictions_test.parquet")
print(pred_df.head())
print("Saved:", len(pred_df), "rows")

      u_id     v_id  y     score
0  c8tx989  c8v3o25  0  0.989961
1  c8uy21q  c8uxg7b  1  0.995518
2  c8v1h6j  c8v1e8x  1  0.990212
3  c8u0yyr  c8tz2yt  1  0.971465
4  c8uaf23  c8ui8p4  0  0.495489
Saved: 3468 rows
